# Reinforcement Learning: A Complete Learning Guide

This notebook teaches **Reinforcement Learning from scratch** before adapting it to your autonomous vehicle project. No prerequisites assumed—we'll learn step by step.

**What you'll learn:**
1. How RL differs from supervised learning
2. Core concepts: agents, environments, rewards, policies
3. The REINFORCE algorithm (policy gradient method)
4. How to implement and train an RL agent
5. How to evaluate learning progress
6. Bridge concepts to your vehicle project

**Why this matters:** Understanding RL fundamentals will help you adapt algorithms to your specific project constraints (4+k sensors, continuous state space, collision penalties, etc.).


## Section 1: Understanding Reinforcement Learning Basics

### What is Reinforcement Learning?

In supervised learning, you show the model examples with correct answers:
```
Input: "What sensorreadings are these?" → Output: "Turn left" (exact answer provided)
```

In **Reinforcement Learning**, the model learns by trial-and-error:
```
Agent makes decision → Environment responds → Agent gets reward/penalty → Agent learns
```

### The Agent-Environment Loop

```
┌──────────────┐                    ┌──────────────────┐
│              │     state (s)      │                  │
│    AGENT     ├──────────────────→ │  ENVIRONMENT    │
│ (Your NN)    │  action (a)        │ (Simulator)      │
│              │ ←──────────────────┤                  │
│              │  reward (r)        │                  │
└──────────────┘                    └──────────────────┘

Step-by-step process:
1. OBSERVE: Agent receives state from environment
2. DECIDE: Agent's neural network chooses an action
3. EXECUTE: Action is applied to the environment
4. REWARD: Environment provides feedback (reward/penalty)
5. LEARN: Agent updates its network based on reward
6. REPEAT: Go to step 1 with new state
```

### Key Terminology

| Term | Meaning | Your Project Example |
|------|---------|----------------------|
| **State (s)** | Current situation | [position_x, position_y, velocity_x, velocity_y, sensor1, sensor2, ...] |
| **Action (a)** | Agent's choice | 0=turn_left, 1=turn_right, 2=speed_up, 3=no_action |
| **Reward (r)** | Feedback signal | +0.025 for speeding up, -0.0125 for turning, -heavy for crash |
| **Policy (π)** | Decision strategy | Neural network that maps states to actions |
| **Trajectory** | One complete game | Sequence of (state, action, reward) tuples |
| **Episode** | One game from start to finish | Car drives until crash or time limit |

### RL vs Supervised Learning: Key Differences

| Aspect | Supervised Learning | Reinforcement Learning |
|--------|-------------------|----------------------|
| **Data** | Get labeled examples directly | Must explore to find good decisions |
| **Feedback** | Immediate, explicit ("right" or "wrong") | Delayed, implicit (reward after many steps) |
| **Exploration** | Not needed | Essential—must try new things to learn |
| **Learning Signal** | Direct error between prediction & truth | Cumulative reward from trajectories |
| **Best When** | You have expert demonstrations | You have a reward function to optimize |


## Section 2: Key RL Concepts—States, Actions, and Rewards

### States (s): The World's Current Situation

A **state** is all the information the agent needs to make a decision at one moment in time.

**For your vehicle project:**
```
State = [x_position, y_position, velocity_x, velocity_y, sensor_1, sensor_2, ..., sensor_k]

Example:
state = [0.5, 0.3, 0.02, 0.01, 0.15, 0.20, 0.10]
         └─────────────────────┘  └────────────────────┘
              Position+Velocity        Sensors (k=3)
```

Why a complete state? So the network can make smart decisions:
- "Where am I?" (position) → "Don't go out of bounds"
- "How fast?" (velocity) → "Not too fast into obstacles"
- "Is there an obstacle left?" (sensors) → "Turn right"

### Actions (a): What the Agent Can Do

An **action** is a choice the agent makes based on current state.

**Actions must be discrete (countable) for your discrete RL agent:**
```
Action space = {0, 1, 2, 3}
  0 → Turn left (angle -= 0.01 rad, speed *= 0.9875)
  1 → Turn right (angle += 0.01 rad, speed *= 0.9875)
  2 → Speed up (speed *= 1.025)
  3 → No action (speed unchanged)
```

The network outputs probabilities for each action, then we sample/choose one.

### Rewards (r): Feedback from the World

A **reward** is a number that tells the agent "was that good or bad?"

```
Positive rewards → Encourage the behavior
Negative rewards → Discourage the behavior

Your vehicle rewards:
  r = +0.025 for "speed up" action
  r = -0.0125 for turning actions
  r = -1.0 for crash (speed drops to 12.5% of current)
  r = (distance moved this frame) - (penalty term) overall score
```

### The Complete Interaction Cycle (Code Perspective)

```python
# Pseudocode of one step:
state = [0.5, 0.3, 0.02, 0.01, 0.15, 0.20, 0.10]  # Current observation

# Agent's neural network makes decision
action_scores = network(state)  # Outputs 4 numbers, one per action
action_probs = softmax(action_scores)  # Convert to probabilities
action = sample(action_probs)  # Pick one action (might pick different ones each time due to randomness)

# Environment executes action
new_state, reward = simulator.step(action)
# For example:
# new_state = [0.502, 0.305, 0.022, 0.010, 0.14, 0.21, 0.12]  (car moved slightly)
# reward = 0.025  (because we chose "speed up")

# Agent remembers this for learning
memory.append({
    'state': state,
    'action': action,
    'reward': reward,
    'next_state': new_state
})
```


## Section 3: Introduction to Policy Gradient Methods

### What's a Policy?

A **policy** is the agent's decision-making strategy. It maps states to actions.

```
Policy π: state → action_probabilities

For your neural network:
π(action | state) = softmax(network_output)
                  = probability distribution over 4 actions
```

### Why Policy Gradient?

There are two main families of RL algorithms:

**Value-based approaches** (Q-learning, DQN):
- Predict: "If I take action X in state S, what total reward will I get?"
- Tricky with continuous or large action spaces
- Indirect policy (derive actions from values)

**Policy Gradient approaches** (REINFORCE, A3C):
- Directly learn: "In state S, which action should I take?"
- More natural for your action space {0,1,2,3}
- Directly optimize the policy
- ✅ **Better for your project**

### The Core Intuition of Policy Gradients

Imagine training a person to play a game:

```
1. Give them the game
2. They play and get a score (reward)
3. Feedback: "You got 100 points!"
4. They remember what they did
5. On the next game, they do more of what worked
```

That's policy gradients! The math:

$$\text{If reward was good} \rightarrow \text{increase probability of the action taken}$$
$$\text{If reward was bad} \rightarrow \text{decrease probability of the action taken}$$

### Why Use Trajectory Rewards?

The key insight: one good reward helps the entire past sequence.

**Example:**
```
Trajectory: state₁ → action₁ → reward +10 (found good speed)
            state₂ → action₂ → reward +10 (avoided crash)
            state₃ → action₃ → reward -1  (hit wall)

Total trajectory reward = 10 + 10 - 1 = 19

Policy update:
  - Increase prob of action₁ (helped earn that +10)
  - Increase prob of action₂ (contributed to good path)
  - Slightly decrease prob of action₃ (at least we got 19 total, not 0)
```

This is more efficient than updating after *every* step. We collect a full trajectory, then update.

### The Learning Signal: Log Probability × Reward

The magic formula that makes policy gradients work:

$$\text{Loss} = -\log(\pi(a|s)) \times R$$

Breaking it down:
- $\log(\pi(a|s))$: How likely was the action we took?
- $R$: What total reward did we get?
- **Negative sign**: We want to *minimize* loss (maximize reward)

**Intuition:**
```
If action was unlikely but worked (high log prob doesn't get large):
  -log(0.1) × 50 = 2.3 × 50 = 115  (big loss to fix)
  
If action was likely and worked (already doing right thing):
  -log(0.8) × 50 = 0.22 × 50 = 11  (small loss to maintain)
  
If any action with bad reward:
  -log(prob) × (-1) = small positive  (gentle penalty)
```

The gradient of this loss tells us how to adjust weights to make good actions more likely!


## Section 4: The REINFORCE Algorithm (Step-by-Step)

### What is REINFORCE?

REINFORCE (**RE**ward **IN**crement = **FORCE**) is the simplest policy gradient algorithm. It's the foundation for many advanced methods.

**Core idea:** "Play the game, collect experiences, use rewards to improve the policy"

### The Algorithm in Pseudocode

```
Initialize neural network weights randomly

For each training iteration:
    1. Collect Trajectories
       - Play N games using current policy
       - For each game, store: (state, action, reward) pairs
       - Calculate total trajectory reward
    
    2. Compute Policy Gradients
       - For each step in each trajectory:
         gradient ← ∇log(π(action | state)) × total_trajectory_reward
    
    3. Update Weights
       - Sum all gradients
       - Backpropagate through network
       - Update weights using optimizer (Adam)
    
    4. Repeat with improved policy
```

### The Algorithm in Mathematical Notation

For a trajectory $\tau = (s_1, a_1, r_1, s_2, a_2, r_2, ..., s_T, a_T, r_T)$:

Step 1 - Calculate total return (cumulative discounted reward):
$$G_t = \sum_{t=0}^{T} \gamma^t r_t$$

where $\gamma$ (gamma) is a discount factor, usually 0.99
- Rewards now matter more than future rewards
- Forces agent to act faster

Step 2 - Policy gradient for each step:
$$\nabla_\theta J(\theta) = \mathbb{E}[\nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t]$$

Step 3 - Update weights:
$$\theta_{new} = \theta_{old} - \alpha \nabla_\theta J(\theta)$$

where $\alpha$ (alpha) is learning rate (0.001 to 0.1 typically)

### Walking Through a Real Example

**Scenario:** Your car has 3 sensors (k=3). Input state: [x=0.5, y=0.3, vx=0.02, vy=0.01, s1=0.15, s2=0.20, s3=0.10]

**Step by step for ONE trajectory:**

```
Frame 1:
  State: [0.5, 0.3, 0.02, 0.01, 0.15, 0.20, 0.10]
  Network outputs: [-0.5, 0.3, 1.2, 0.1]  (raw scores)
  Softmax (probs): [0.08, 0.12, 0.75, 0.05]  (normalized to sum=1)
  Sample action: 2 (speed up, chosen with 75% probability)
  Reward: +0.025
  Memory: store (state, action=2, reward=0.025)

Frame 2:
  State: [0.502, 0.305, 0.022, 0.010, 0.14, 0.21, 0.12]
  Network outputs: [0.2, -0.4, 0.8, 0.0]
  Softmax (probs): [0.22, 0.10, 0.54, 0.14]
  Sample action: 2 (speed up again)
  Reward: +0.025
  Memory: add (state, action=2, reward=0.025)

... game continues ...

Frame 10 (last frame before crash):
  State: [0.8, 0.8, 0.05, 0.05, 0.05, 0.10, 0.15]
  Network samples action: 0 (turn left)
  Reward: -1.0 (crash!)
  Memory: add (state, action=0, reward=-1.0)

Trajectory complete. Total reward = 0.025×9 - 1.0 = -0.775

Now update network:
  For each step in trajectory:
    log_prob_of_action = log(probability of action we took)
    
    Step 1: log(0.75) × (-0.775) = -0.28 × (-0.775) ≈ 0.22
    Step 2: log(0.54) × (-0.775) = -0.62 × (-0.775) ≈ 0.48
    ...
    Step 10: log(0.10) × (-0.775) = -2.30 × (-0.775) ≈ 1.78
    
  Sum all: total_loss ≈ some value
  Backpropagate: tells us how much each weight contributed to bad loss
  Update weights: make good actions more likely, bad actions less likely
```

### Key Parameters to Understand

| Parameter | Role | Typical Value |
|-----------|------|---------------|
| **Learning Rate** | How big weight updates are | 0.001 - 0.01 |
| **Discount Factor (γ)** | How much future matters | 0.99 (present matters most) |
| **Episodes Per Batch** | Games before one weight update | 10-50 |
| **Epochs** | Number of times to do batch updates | 100-1000+ |
| **Batch Size** | Games played in parallel (if available) | 10-32 |

### Why This Works: The Intuition

```
Good trajectory → total_reward is positive
    → -log(prob) × positive = encourages high probability for those actions
    → Weights shift to make those actions more likely in similar states

Bad trajectory → total_reward is negative
    → -log(prob) × negative = encourages low probability for those actions  
    → Weights shift to make those actions less likely

Repeated many times → policy improves!
```


## Section 5: Simple RL Environment Setup

Before we use a complex simulator, let's practice with a **simple gridworld environment** that teaches RL principles.

**Why:** Simpler environments train faster and are easier to debug.

### Simple Environment: "Reach the Goal"

```
Grid: 5x5
Start: [0, 0] (bottom-left)
Goal: [4, 4] (top-right)
Actions: 0=up, 1=right, 2=down, 3=left
Rewards: +1 for reaching goal, -0.1 for each step (encourage speed)
```

This is actually similar to the existing code in PolicyNetwork.ipynb, but we'll add RL training to it.

### Environment Code: State Representation

```python
# State format: [x, y] for position
# Simple: just need 2 numbers

# Action effects:
action_effects = {
    0: (0, 1),   # Up: increase y
    1: (1, 0),   # Right: increase x
    2: (0, -1),  # Down: decrease y
    3: (-1, 0)   # Left: decrease x
}

# Rewards:
reward_dict = {
    'goal_reached': 1.0,      # Win!
    'step': -0.1,             # Cost per step (pushes toward efficiency)
    'out_of_bounds': -0.5,    # Penalty for hitting walls
}

# Episode ends when:
# 1. Reached goal (done = True)
# 2. Took > 50 steps (done = True, return -1.0)
```

### Why This Teaches RL Well

✅ **Small input size** (just 2 numbers) → Fast training to test concepts
✅ **Clear reward signal** (goal/steps) → Easy to understand behavior
✅ **Optimal solution exists** (move right then up) → Can evaluate accuracy
✅ **Deterministic** → Same input always gives same result (vs random environment)

Later, we'll adapt this to your continuous car space (add velocity, sensors, crashes).


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# ============================================================================
# SIMPLE ENVIRONMENT: Grid World (Learning Purposes)
# ============================================================================

class SimpleGridEnv:
    """
    Simple 5x5 gridworld environment for learning RL.
    
    State: [x, y] position (2 numbers)
    Actions: 0=up, 1=right, 2=down, 3=left
    Goal: Reach [4, 4] from [0, 0]
    """
    
    def __init__(self, width=5, height=5):
        self.width = width
        self.height = height
        self.goal = [width - 1, height - 1]  # Top-right corner
        self.start = [0, 0]  # Bottom-left
        self.state = self.start.copy()
        self.step_count = 0
        self.max_steps = 50
    
    def reset(self):
        """Reset environment to start position."""
        self.state = self.start.copy()
        self.step_count = 0
        return self.state
    
    def step(self, action):
        """
        Execute action and return (next_state, reward, done).
        
        Args:
            action: int in {0, 1, 2, 3}
                0 = up (y+1)
                1 = right (x+1)
                2 = down (y-1)
                3 = left (x-1)
        
        Returns:
            next_state: [x, y]
            reward: float
            done: bool (episode finished)
        """
        self.step_count += 1
        x, y = self.state
        
        # Apply action
        if action == 0:    # Up
            y += 1
        elif action == 1:  # Right
            x += 1
        elif action == 2:  # Down
            y -= 1
        elif action == 3:  # Left
            x -= 1
        
        # Clamp to boundaries
        x = max(0, min(x, self.width - 1))
        y = max(0, min(y, self.height - 1))
        self.state = [x, y]
        
        # Calculate reward
        reward = -0.1  # Small penalty per step (encourages efficiency)
        
        # Check termination conditions
        done = False
        if self.state == self.goal:
            reward = 1.0  # Big reward for reaching goal!
            done = True
        elif self.step_count >= self.max_steps:
            reward = -1.0  # Penalty for taking too long
            done = True
        
        return self.state, reward, done


# ============================================================================
# NEURAL NETWORK POLICY
# ============================================================================

class PolicyNetwork(nn.Module):
    """
    Simple neural network policy for RL.
    
    Architecture:
    Input (state) → Hidden 1 (ReLU) → Hidden 2 (ReLU) → Output (4 actions)
    """
    
    def __init__(self, state_size, hidden_size=64, action_size=4):
        """
        Initialize network.
        
        Args:
            state_size: dimension of state (e.g., 2 for [x, y])
            hidden_size: neurons in hidden layers
            action_size: number of actions (4 for our grid)
        """
        super(PolicyNetwork, self).__init__()
        
        # Layer 1: state → hidden
        self.layer1 = nn.Linear(state_size, hidden_size)
        self.relu1 = nn.ReLU()
        
        # Layer 2: hidden → hidden
        self.layer2 = nn.Linear(hidden_size, hidden_size)
        self.relu2 = nn.ReLU()
        
        # Output layer: hidden → action scores
        self.output = nn.Linear(hidden_size, action_size)
    
    def forward(self, state):
        """
        Forward pass: state → action probabilities.
        
        Args:
            state: tensor of shape (state_size,) or (batch, state_size)
        
        Returns:
            action_scores: tensor of shape (4,) with raw scores for each action
        """
        x = self.layer1(state)
        x = self.relu1(x)
        x = self.layer2(x)
        x = self.relu2(x)
        action_scores = self.output(x)
        return action_scores


# ============================================================================
# TEST: Verify environment and network work
# ============================================================================

print("=" * 60)
print("TEST 1: Environment Setup")
print("=" * 60)

env = SimpleGridEnv(width=5, height=5)
print(f"Initial state: {env.reset()}")
print(f"Goal position: {env.goal}")

# Test a few steps
for i in range(3):
    action = i % 4
    next_state, reward, done = env.step(action)
    print(f"  Action {action}: state={next_state}, reward={reward:.2f}, done={done}")

print("\n" + "=" * 60)
print("TEST 2: Neural Network")
print("=" * 60)

network = PolicyNetwork(state_size=2, hidden_size=64, action_size=4)
state = [0.0, 0.0]
state_tensor = torch.FloatTensor(state)

print(f"Input state: {state}")
action_scores = network(state_tensor)
print(f"Network output (action scores): {action_scores}")

# Convert to probabilities
action_probs = torch.softmax(action_scores, dim=0)
print(f"After softmax (probabilities): {action_probs}")
print(f"Sum of probabilities: {action_probs.sum():.4f} (should be ~1.0)")

print("\nAll tests passed! Ready for training.")


## Section 6: Training Your First RL Agent with REINFORCE

Now we implement the complete REINFORCE algorithm. Let's build it step-by-step.

### The REINFORCE Training Algorithm (Code Version)

We'll implement three functions:

1. **`collect_trajectory()`** - Play one game, record decisions
2. **`compute_returns()`** - Calculate cumulative rewards for each step
3. **`train_step()`** - Update network weights based on trajectories


In [ ]:
# ============================================================================
# REINFORCE ALGORITHM COMPONENTS
# ============================================================================

def collect_trajectory(network, env):
    """
    Play one complete game using current policy.
    Record: state, action, reward, log_prob for each step.
    
    Args:
        network: PolicyNetwork instance
        env: SimpleGridEnv instance
    
    Returns:
        trajectory: list of dicts with keys:
            - 'state': current state [x, y]
            - 'action': action taken (0-3)
            - 'reward': reward received
            - 'log_prob': log(probability) of the action we took
    """
    trajectory = []
    state = env.reset()
    done = False
    
    while not done:
        # Convert state to tensor
        # Why: PyTorch networks work with tensors, not Python lists
        state_tensor = torch.FloatTensor(state)
        
        # Get action scores from network
        # Shape: (4,) with one score per action
        action_scores = network(state_tensor)
        
        # Convert scores to probabilities
        # Softmax: exponential of each score / sum of exponentials
        # Why: ensures all probabilities sum to 1
        action_probs = torch.softmax(action_scores, dim=0)
        
        # Sample action from probability distribution
        # This introduces randomness (exploration)
        # Why: helps agent explore different actions, not just greedy
        action_distribution = torch.distributions.Categorical(action_probs)
        action = action_distribution.sample()
        
        # Calculate log probability of the action we just took
        # This is used later for the policy gradient
        log_prob = torch.log(action_probs[action])
        
        # Execute action in environment
        next_state, reward, done = env.step(action.item())
        
        # Store experience
        trajectory.append({
            'state': state,
            'action': action.item(),  # Convert to Python int
            'reward': reward,
            'log_prob': log_prob  # Keep as tensor, we need this for gradients
        })
        
        state = next_state
    
    return trajectory


def compute_returns(trajectory, gamma=0.99):
    """
    Calculate discounted cumulative returns for each step.
    
    This is crucial: we assign credit to earlier actions based on final outcome.
    
    Example:
        trajectory: rewards = [0.1, 0.1, -0.1, 1.0]
        
        Returns:
        G_t = sum of future rewards weighted by gamma
        G_3 = 1.0                              (only final reward)
        G_2 = -0.1 + 0.99 * 1.0 = 0.89        (future rewards discounted)
        G_1 = 0.1 + 0.99 * 0.89 = 0.989       (more future discounting)
        G_0 = 0.1 + 0.99 * 0.989 = 1.088      (all future rewards)
    
    Args:
        trajectory: list of dicts with 'reward' key
        gamma: discount factor (0.99)
    
    Returns:
        returns: list of cumulative returns for each step
    """
    returns = []
    cumulative_return = 0.0
    
    # Iterate backward through trajectory
    # Why backward: easier to do running sum
    for step in reversed(trajectory):
        reward = step['reward']
        # Bellman equation: G_t = r_t + gamma * G_{t+1}
        cumulative_return = reward + gamma * cumulative_return
        returns.insert(0, cumulative_return)
    
    return returns


def train_step(network, optimizer, batch_trajectories, gamma=0.99):
    """
    Perform one REINFORCE update on a batch of trajectories.
    
    This is the heart of policy gradient learning.
    
    Args:
        network: PolicyNetwork to update
        optimizer: torch.optim.Adam or similar
        batch_trajectories: list of trajectories (each from collect_trajectory)
        gamma: discount factor
    """
    
    # Step 1: Clear old gradients
    # Why: gradients accumulate by default; we want fresh gradients each step
    optimizer.zero_grad()
    
    # Step 2: Calculate loss over all trajectories
    total_loss = 0
    
    for trajectory in batch_trajectories:
        # Calculate returns (cumulative discounted rewards)
        returns = compute_returns(trajectory, gamma)
        
        # For each step in this trajectory
        for step, G_t in zip(trajectory, returns):
            log_prob = step['log_prob']
            
            # The REINFORCE update rule:
            # loss = -log_prob * G_t
            # 
            # Why negative? 
            # - We want to maximize reward (increase log_prob when reward good)
            # - But PyTorch minimizes loss
            # - So we minimize negative log_prob (equivalent to maximizing log_prob)
            #
            # Why multiply by G_t?
            # - Good trajectory (G_t > 0): encourages high log_prob (high probability)
            # - Bad trajectory (G_t < 0): encourages low log_prob (low probability)
            # - Neutral trajectory (G_t ≈ 0): minimal update
            loss = -log_prob * G_t
            
            # Accumulate loss
            total_loss += loss
    
    # Step 3: Backpropagation
    # Compute gradient of loss with respect to all network weights
    # This tells us: "which weights caused this loss? how much?"
    total_loss.backward()
    
    # Step 4: Update weights using optimizer
    # optimizer.step() applies gradient descent: w_new = w_old - lr * gradient
    optimizer.step()
    
    # Return loss for tracking (lower is better)
    return total_loss.item()


# ============================================================================
# TRAINING LOOP
# ============================================================================

print("\n" + "=" * 60)
print("COMPLETE REINFORCE TRAINING")
print("=" * 60)

# Hyperparameters
state_size = 2
hidden_size = 64
action_size = 4
learning_rate = 0.01  # How big weight updates are
num_epochs = 100  # How many times we update
games_per_epoch = 10  # How many games before each update
gamma = 0.99  # Discount factor
print(f"\nHyperparameters:")
print(f"  Learning rate: {learning_rate}")
print(f"  Hidden layer size: {hidden_size}")
print(f"  Games per epoch: {games_per_epoch}")
print(f"  Total epochs: {num_epochs}")

# Initialize environment and network
env = SimpleGridEnv(width=5, height=5)
network = PolicyNetwork(state_size, hidden_size, action_size)
optimizer = optim.Adam(network.parameters(), lr=learning_rate)

# Track metrics
losses = []
trajectory_rewards = []

# Training loop
print(f"\nStarting training...")
for epoch in range(num_epochs):
    
    # Step 1: Collect batch of trajectories
    batch_trajectories = []
    epoch_rewards = []
    
    for game_num in range(games_per_epoch):
        trajectory = collect_trajectory(network, env)
        batch_trajectories.append(trajectory)
        
        # Calculate total reward for this game
        total_reward = sum(step['reward'] for step in trajectory)
        epoch_rewards.append(total_reward)
    
    # Step 2: Train network on this batch
    loss = train_step(network, optimizer, batch_trajectories, gamma)
    
    # Step 3: Track metrics
    losses.append(loss)
    avg_reward = np.mean(epoch_rewards)
    trajectory_rewards.append(avg_reward)
    
    # Step 4: Print progress
    if epoch % 10 == 0 or epoch == num_epochs - 1:
        print(f"Epoch {epoch:3d}/{num_epochs}: Loss={loss:.4f}, "
              f"Avg Reward={avg_reward:.4f}, "
              f"Reward Range=[{min(epoch_rewards):.4f}, {max(epoch_rewards):.4f}]")

print("\nTraining complete!")
print(f"Final average reward: {trajectory_rewards[-1]:.4f}")
print(f"Initial average reward: {trajectory_rewards[0]:.4f}")
print(f"Improvement: {trajectory_rewards[-1] - trajectory_rewards[0]:.4f}")


## Section 7: Evaluating Agent Performance

### Visualization: Did the Agent Learn?

We'll create plots to understand training progress.


In [ ]:
import matplotlib.pyplot as plt

# ============================================================================
# EVALUATION: Test trained agent
# ============================================================================

print("\n" + "=" * 60)
print("EVALUATION: Testing Trained Agent")
print("=" * 60)

def evaluate_agent(network, env, num_games=20):
    """
    Run trained agent for multiple games and report statistics.
    
    Args:
        network: trained PolicyNetwork
        env: environment
        num_games: games to test
    
    Returns:
        rewards: list of total rewards per game
        trajectories: list of trajectories
    """
    
    rewards = []
    trajectories = []
    
    for _ in range(num_games):
        trajectory = []
        state = env.reset()
        done = False
        total_reward = 0
        
        while not done:
            # Use greedy policy (pick action with highest probability, no randomness)
            state_tensor = torch.FloatTensor(state)
            action_scores = network(state_tensor)
            action_probs = torch.softmax(action_scores, dim=0)
            
            # argmax: pick action with highest probability
            action = torch.argmax(action_probs).item()
            
            next_state, reward, done = env.step(action)
            trajectory.append({
                'state': state,
                'action': action,
                'reward': reward
            })
            total_reward += reward
            state = next_state
        
        rewards.append(total_reward)
        trajectories.append(trajectory)
    
    return rewards, trajectories


# Test the agent
test_rewards, test_trajectories = evaluate_agent(network, env, num_games=20)

print(f"\nEvaluation Results (20 games):")
print(f"  Average reward: {np.mean(test_rewards):.4f}")
print(f"  Std deviation: {np.std(test_rewards):.4f}")
print(f"  Best game: {max(test_rewards):.4f}")
print(f"  Worst game: {min(test_rewards):.4f}")
print(f"  Successful (reward > 0.5): {sum(1 for r in test_rewards if r > 0.5)}/20")

# ============================================================================
# VISUALIZATION: Plot training curves
# ============================================================================

print("\nGenerating visualizations...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Training loss
ax1 = axes[0]
ax1.plot(losses, linewidth=2, color='blue')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training Loss Over Time\n(Should generally decrease)', fontsize=12)
ax1.grid(True, alpha=0.3)

# Plot 2: Average reward per epoch
ax2 = axes[1]
ax2.plot(trajectory_rewards, linewidth=2, color='green')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Average Reward', fontsize=12)
ax2.set_title('Average Trajectory Reward\n(Should generally increase)', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Training curves saved to 'training_curves.png'")

# ============================================================================
# ANALYSIS: What did the agent learn?
# ============================================================================

print("\n" + "=" * 60)
print("ANALYSIS: What Action Does Agent Choose in Each State?")
print("=" * 60)

action_names = ['UP', 'RIGHT', 'DOWN', 'LEFT']
print("\nAgent's greedy policy for each grid position:")
print("(Shows which action the network chooses)")
print()

policy_grid = [[None for _ in range(5)] for _ in range(5)]

for x in range(5):
    for y in range(5):
        state = [x, y]
        state_tensor = torch.FloatTensor(state)
        action_scores = network(state_tensor)
        best_action = torch.argmax(action_scores).item()
        policy_grid[y][x] = best_action

# Print policy grid
print("  x: 0    1    2    3    4")
for y in range(4, -1, -1):
    row_str = f"y:{y} "
    for x in range(5):
        action = policy_grid[y][x]
        action_symbol = ['↑', '→', '↓', '←'][action] if action is not None else '?'
        row_str += f" {action_symbol} "
    print(row_str)

print("\nInterpretation:")
print("  ↑ = UP, → = RIGHT, ↓ = DOWN, ← = LEFT")
print("  Goal: Agent should learn RIGHT and UP to reach [4,4] from [0,0]")

# Check if policy makes sense
expected_actions = {
    # Bottom-left corner should go right or up
    (0, 0): [0, 1],  # UP or RIGHT
    # Top edge (except corner) should go right
    (0, 4): [1],  # RIGHT
    (2, 4): [1],  # RIGHT
    # Right edge (except corner) should go up
    (4, 0): [0],  # UP
    (4, 2): [0],  # UP
}

print("\nPolicy Correctness Check:")
correct = 0
for (x, y), acceptable_actions in expected_actions.items():
    actual = policy_grid[y][x]
    if actual in acceptable_actions:
        print(f"  ✓ Position [{x},{y}]: chose {action_names[actual]} (good)")
        correct += 1
    else:
        print(f"  ✗ Position [{x},{y}]: chose {action_names[actual]} (suboptimal)")

print(f"\nCorrectness: {correct}/{len(expected_actions)}")

if correct == len(expected_actions):
    print("🎉 Agent learned optimal policy!")
else:
    print("⚠ Agent may need more training or tuning")


## Section 8: Bridge to Your Autonomous Vehicle Project

### How Does This Connect?

**What we built:**
- Simple 5×5 gridworld with REINFORCE ✓
- Agent learns to reach goal efficiently ✓
- We can visualize learned policy ✓

**Your actual project requirements:**
- Continuous 2D space [0,1]×[0,1] (not discrete grid)
- 4+k sensors (not just position)
- Realistic physics (velocity, speed, crashes)
- Competitive scoring (distance traveled)

### Key Adaptations Needed

| Aspect | Learning (This Notebook) | Your Project |
|--------|--------------------------|--------------|
| **State** | [x, y] (2 values) | [x, y, vx, vy, s1, ..., sk] (6+k values) |
| **Action Space** | {0,1,2,3} discrete | {0,1,2,3} discrete (same!) |
| **Rewards** | +1 goal, -0.1 per step | +0.025 speed, -0.0125 turn, -crash |
| **Dynamics** | Instant movement | Velocity-based, inertia |
| **Termination** | Goal reached or 50 steps | 50 steps or crash |
| **Optimization** | Supervised training | Online learning from simulator |

### Architecture Changes

**Hidden layer sizes:**
- Learning: 64 neurons (small, trains fast)
- Your project: up to 100 neurons (larger, more complex)

**Input layer:**
- Learning: 4 + 0 = 4 (just position × 2 + dummy × 2)
- Your project: 4 + k where k ∈ {1, 3, 5, 7, 9}

**The code structure stays the same!**
- Still use REINFORCE algorithm
- Still collect trajectories
- Still use softmax + categorical distribution
- Still update with policy gradients

### Your Next Steps (After Understanding This Guide)

1. **Load the simulator** - Use the demo environment provided
2. **Adapt the environment** - Create wrapper for simulator with your reward function
3. **Adjust input size** - Change from 2 → 4+k in PolicyNetwork
4. **Update rewards** - Change from goal-based → distance-based
5. **Add sensor processing** - Parse k sensor values from state
6. **Scale hyperparameters** - More hidden neurons, more training
7. **Test incrementally** - Start with k=1, then 3, 5, etc.

### Questions to Ask Yourself

✓ "What state information does my agent need?" → Helps you design input size
✓ "What actions should it take?" → Confirms action space {0,1,2,3}
✓ "What reward encourages good behavior?" → Helps you write reward function
✓ "How do I know if training is working?" → Collect metrics like you see here

### Common Debugging When Adapting

**Problem:** Network output has NaN
- **Cause:** Sensor values too large or reward signal too extreme
- **Fix:** Normalize inputs, reward scaling

**Problem:** Agent always does same action
- **Cause:** Rewards too uniform, network not learning
- **Fix:** Increase reward difference between good/bad choices

**Problem:** Training is very slow
- **Cause:** Learning rate too small, hidden layers too big
- **Fix:** Increase learning rate, or decrease hidden layer size

---

## Key Takeaways

✅ **RL Core:** Agent learns by receiving rewards for actions, then adjusts policy
✅ **REINFORCE:** Simple policy gradient method—collect trajectories, update based on cumulative rewards
✅ **Exploration vs Exploitation:** Use softmax + sampling for exploration, argmax for testing
✅ **Policy Representation:** Neural network directly maps states to action probabilities
✅ **Scaling:** Same algorithm works for simple grids AND complex continuous environments

You're ready to apply this to your vehicle project! The concepts are universal—only the environment changes.


## Quick Reference: RL Formulas & Code Patterns

### Core Equations

**Softmax (convert scores to probabilities):**
$$\pi_\theta(a|s) = \frac{e^{s_a}}{\sum_i e^{s_i}}$$

**Categorical sampling (pick action):**
$$a \sim \text{Categorical}(\pi_\theta(a|s))$$

**Discounted return (cumulative reward):**
$$G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + ... = \sum_{k=0}^{T-t} \gamma^k r_{t+k}$$

**REINFORCE loss (for one step):**
$$\mathcal{L}_t = -\log \pi_\theta(a_t | s_t) \cdot G_t$$

**Gradient update:**
$$\theta \leftarrow \theta - \alpha \nabla_\theta \mathcal{L}$$

### Essential Code Patterns

**Pattern 1: Forward pass (get probabilities)**
```python
state_tensor = torch.FloatTensor(state)
action_scores = network(state_tensor)  # Shape: (4,)
action_probs = torch.softmax(action_scores, dim=0)  # Normalize
```

**Pattern 2: Sample action (training)**
```python
action_distribution = torch.distributions.Categorical(action_probs)
action = action_distribution.sample()  # Random, includes exploration
log_prob = torch.log(action_probs[action])  # For gradient update
```

**Pattern 3: Greedy action (evaluation)**
```python
action = torch.argmax(action_probs).item()  # Deterministic
```

**Pattern 4: REINFORCE update**
```python
optimizer.zero_grad()
loss = -log_prob * cumulative_return  # Negative: we minimize
loss.backward()
optimizer.step()
```

### Hyperparameter Guide

| Parameter | Range | Effect |
|-----------|-------|---------|
| Learning rate (α) | 0.0001 - 0.1 | Higher = faster but unstable; lower = slow |
| Hidden size | 32 - 256 | Larger = more expressiveness, slower training |
| Discount (γ) | 0.95 - 0.99 | Closer to 1 = care more about future rewards |
| Games/batch | 10 - 100 | More = stable, slower; less = noisy, faster |
| Epochs | 100 - 10000 | More = better learning, longer time |

### Debugging Checklist

- [ ] Network output shape is (4,)?
- [ ] Softmax probabilities sum to ~1.0?
- [ ] Can sample different actions (not always same)?
- [ ] Loss decreases over epochs?
- [ ] Rewards generally improve (trending up)?
- [ ] Agent explores different states (visits different positions)?
- [ ] No NaN or Inf values?
- [ ] Evaluation reward > training baseline?

---

## Glossary

| Term | Definition |
|------|-----------|
| **Trajectory** | One complete episode: sequence of (s, a, r) tuples |
| **Policy (π)** | Function mapping states to action probabilities |
| **Return (G)** | Cumulative discounted reward from a step onward |
| **On-policy** | Learning from actions your current policy takes (like REINFORCE) |
| **Off-policy** | Learning from actions other policies took (like Q-learning) |
| **Exploration** | Trying random actions to discover new behaviors |
| **Exploitation** | Using actions you know are rewarding |
| **Gradient** | Direction to change weights to reduce loss |
| **Log probability** | log of probability; used in policy gradients |
